# 01 — Data Ingestion

## CSV → LangChain Documents

**Experiment ID:** ING-001

### Objective

Load the canonical Pharma Sales CSV dataset into LangChain and understand how CSV records are represented as LangChain `Document` objects.

### Questions We Want to Answer

1. How many records are present in the dataset?
2. How does `CSVLoader` represent a CSV record?
3. What is stored in `page_content`?
4. What is stored in `metadata`?
5. What is the typical size of each document?

### Expected Outcome

By the end of this notebook, we will have:

- Loaded the canonical CSV dataset
- Converted CSV records into LangChain `Document` objects
- Understood `page_content` vs `metadata`
- Measured baseline document sizes
- Validated the ingestion process
- Established the baseline for the chunking experiment

### Pipeline Position

```text
CSV File
   ↓
CSVLoader
   ↓
LangChain Documents
   ↓
Document Validation
   ↓
Document Statistics
   ↓
Chunking Experiments
```

### Experiment Principle

This notebook establishes the **data baseline**.

No transformations are applied to the document content at this stage.

## 1. Environment & Imports

The project uses environment variables for configuration such as the OpenAI API key.

Although this notebook does not call an OpenAI service directly, validating the project environment here ensures that the same environment can be used consistently across the downstream notebooks.

In [22]:
# Standard library
import os

# Environment configuration
from dotenv import load_dotenv

# LangChain document loader
from langchain_community.document_loaders import CSVLoader


# Load environment variables from the local .env file.
load_dotenv()

# Validate that the API key required by downstream notebooks is available.
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY environment variable is not set. "
        "Please configure it in your .env file."
    )

print("Environment configured successfully.")

Environment configured successfully.


## 2. Dataset Configuration

The project contains multiple files under the `data/` directory.

For reproducible experimentation, this notebook intentionally loads only the canonical Pharma Sales dataset.

This prevents supporting files such as `sample_questions.csv` from accidentally becoming part of the ingestion baseline.

In [23]:
# Canonical dataset used throughout the CSV RAG experiments.
DATA_PATH = "../data/Pharma_Sales_Long.csv"

print(f"Dataset: {DATA_PATH}")

Dataset: ../data/Pharma_Sales_Long.csv


## 3. Load CSV Data

`CSVLoader` converts each row of the CSV file into a LangChain `Document`.

For this project:

```text
CSV Row
   ↓
LangChain Document
```

Therefore:

```text
300 CSV Records
       ↓
300 LangChain Documents
```

Each `Document` contains:

- `page_content` — textual representation of the CSV record
- `metadata` — information describing the source record

In [24]:
# CSVLoader converts each CSV row into a LangChain Document.
loader = CSVLoader(
    file_path=DATA_PATH,
    encoding="utf-8"
)

data = loader.load()

print(f"Loaded {len(data)} documents.")

Loaded 300 documents.


## 4. Validate Data Ingestion

Before using the documents in downstream experiments, we perform basic validation.

The purpose is to catch ingestion problems early, such as:

- No documents being loaded
- Empty document content
- Missing metadata

In [25]:
# Basic ingestion validation.
assert len(data) > 0, "No documents were loaded."

assert all(
    document.page_content.strip()
    for document in data
), "One or more documents contain empty page_content."

assert all(
    document.metadata
    for document in data
), "One or more documents are missing metadata."

print("✓ Ingestion validation passed.")
print(f"✓ Documents validated: {len(data)}")

✓ Ingestion validation passed.
✓ Documents validated: 300


## 5. Understand the Loaded Data

`CSVLoader.load()` returns a Python list.

Each element in the list is a LangChain `Document`.

Conceptually:

```text
CSV
 │
 ├── Row 1 ──→ Document
 ├── Row 2 ──→ Document
 ├── Row 3 ──→ Document
 │
 └── Row N ──→ Document
```

The two fields we are primarily interested in are:

### `page_content`

Contains the textual representation of the CSV record.

### `metadata`

Contains information about the origin of the document, such as the source file and row number.

In [26]:
print(f"Container type : {type(data)}")
print(f"Document type  : {type(data[0])}")

Container type : <class 'list'>
Document type  : <class 'langchain_core.documents.base.Document'>


## 6. Inspect a Sample Document

Let's inspect the first document in detail.

This helps us understand what information will be passed into the downstream chunking and embedding stages.

In [27]:
sample_document = data[0]

print("PAGE CONTENT")
print("=" * 80)
print(sample_document.page_content)

print("\nMETADATA")
print("=" * 80)
print(sample_document.metadata)

PAGE CONTENT
Transaction_ID: TXN00001
Product: GARDASIL 9
Business_Unit: Vaccines
Indication: HPV Prevention
Region: West
State: Maharashtra
Territory: T001
HCP_Speciality: Surgeon
Sales_Representative: Rep_1
Quantity: 12
Sales_Value: 661097
Transaction_Date: 2026-01-17
Notes: Product Overview: GARDASIL 9 is used in HPV Prevention. Clinical discussion covered approved indications, patient eligibility, efficacy, safety, monitoring, treatment pathway, and physician education. Sales discussion included customer questions, objections, market access, competitor comparison, follow-up planning, CRM updates, scientific literature sharing, compliant promotion, and future engagement opportunities. Territory insights included prescription trends, customer behavior, business opportunities, formulary discussions, regional planning, call objectives, meeting outcomes, and action items. Product Overview: GARDASIL 9 is used in HPV Prevention. Clinical discussion covered approved indications, patient el

### Understanding `page_content` vs `metadata`

The distinction between `page_content` and `metadata` is important for the rest of the project.

### `page_content`

Contains the actual business information from the CSV record.

Example:

```text
Product: GARDASIL 9
Business_Unit: Vaccines
Indication: HPV Prevention
Region: West
...
```

This content will later flow through:

```text
Document
   ↓
Chunk
   ↓
Embedding
   ↓
Vector Store
   ↓
Similarity Search
```

### `metadata`

Contains information about the source of the document.

Example:

```python
{
    "source": "../data/Pharma_Sales_Long.csv",
    "row": 0
}
```

Metadata allows us to identify the original source record after retrieval.

### Key Takeaway

**`page_content` answers: "What does this document contain?"**

**`metadata` answers: "Where did this document come from?"**

## 7. Inspect Multiple Documents

We will inspect only the first three records.

Displaying all 300 records would create unnecessary notebook output and make the experiment harder to review.

In [28]:
for index, document in enumerate(data[:3], start=1):
    print("=" * 80)
    print(f"Document {index}")
    print(f"Metadata: {document.metadata}")
    print("Content preview:")
    print(document.page_content[:300])
    print("...")

Document 1
Metadata: {'source': '../data/Pharma_Sales_Long.csv', 'row': 0}
Content preview:
Transaction_ID: TXN00001
Product: GARDASIL 9
Business_Unit: Vaccines
Indication: HPV Prevention
Region: West
State: Maharashtra
Territory: T001
HCP_Speciality: Surgeon
Sales_Representative: Rep_1
Quantity: 12
Sales_Value: 661097
Transaction_Date: 2026-01-17
Notes: Product Overview: GARDASIL 9 is use
...
Document 2
Metadata: {'source': '../data/Pharma_Sales_Long.csv', 'row': 1}
Content preview:
Transaction_ID: TXN00002
Product: WELIREG
Business_Unit: Oncology
Indication: Renal Cell Carcinoma
Region: North
State: Tamil Nadu
Territory: T002
HCP_Speciality: Oncologist
Sales_Representative: Rep_2
Quantity: 3
Sales_Value: 627814
Transaction_Date: 2026-07-02
Notes: Product Overview: WELIREG is u
...
Document 3
Metadata: {'source': '../data/Pharma_Sales_Long.csv', 'row': 2}
Content preview:
Transaction_ID: TXN00003
Product: ZEPATIER
Business_Unit: Infectious Disease
Indication: Hepatitis C
Region: South

## 8. Document Size Analysis

Before chunking the documents, we need to understand their current size.

The document length provides a baseline for deciding which chunk sizes should be evaluated in Notebook 02.

If documents are significantly larger than the selected chunk size, a single document will be divided into multiple chunks.

In [29]:
# Measure the textual size of every loaded document.
doc_lengths = [
    len(document.page_content)
    for document in data
]

min_length = min(doc_lengths)
max_length = max(doc_lengths)
avg_length = sum(doc_lengths) / len(doc_lengths)

print("DATASET SUMMARY")
print("=" * 60)
print(f"Total Documents       : {len(data)}")
print(f"Minimum Length        : {min_length} characters")
print(f"Maximum Length        : {max_length} characters")
print(f"Average Length        : {avg_length:.2f} characters")

DATASET SUMMARY
Total Documents       : 300
Minimum Length        : 2031 characters
Maximum Length        : 2142 characters
Average Length        : 2084.79 characters


## 9. Experiment Findings

### Results

| Metric | Result |
|---|---:|
| Total Documents | 300 |
| Minimum Document Length | 2,031 characters |
| Maximum Document Length | 2,142 characters |
| Average Document Length | 2,084.79 characters |

### Observations

1. Each CSV record is successfully represented as a LangChain `Document`.
2. The dataset contains 300 documents.
3. Document lengths are relatively consistent.
4. The average document length is approximately 2,085 characters.
5. Source information is retained in document metadata.

### Key Learning

The ingestion stage converts structured CSV records into a document-oriented representation suitable for downstream LangChain processing.

### Baseline

**Average document length ≈ 2,085 characters**

This value will be used as the baseline when evaluating chunking strategies.

## 10. Decision

### Decision: Proceed to Chunking Experiments

The ingestion configuration is accepted as the project baseline.

The next stage will investigate how different values of:

- `chunk_size`
- `chunk_overlap`

affect document fragmentation.

### Initial Chunking Experiments

| Experiment | Chunk Size | Chunk Overlap |
|---|---:|---:|
| CHK-001 | 250 | 20 |
| CHK-002 | 500 | 50 |
| CHK-003 | 1,000 | 100 |

### Hypothesis

Smaller chunks are expected to produce more chunks, while larger chunks should produce fewer chunks.

However, the objective is not simply to minimize the number of chunks.

The final configuration should provide a useful balance between:

- Retrieval precision
- Context preservation
- Number of chunks
- Downstream retrieval performance

## 11. Handoff to Notebook 02

The ingestion stage is complete.

```text
300 CSV Records
       ↓
300 LangChain Documents
       ↓
Average Document Size ≈ 2,085 characters
       ↓
Chunking Experiments
```

Notebook 02 will use the same canonical dataset and evaluate multiple chunking configurations.

The selected chunking configuration will become the baseline for the embedding and retrieval experiments.

In [30]:
# Final notebook-level validation.
assert len(data) == 300, (
    f"Expected 300 documents, but found {len(data)}."
)

assert min_length > 0
assert max_length >= min_length
assert min_length <= avg_length <= max_length

print("✓ Notebook 01 completed successfully.")
print(f"✓ Canonical documents: {len(data)}")
print(f"✓ Average document length: {avg_length:.2f} characters")
print("✓ Ready for Notebook 02 — Chunking Experiments")

✓ Notebook 01 completed successfully.
✓ Canonical documents: 300
✓ Average document length: 2084.79 characters
✓ Ready for Notebook 02 — Chunking Experiments
